In [ ]:
from collections import Counter, defaultdict
from dataclasses import dataclass
import functools
from io import StringIO
import itertools
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.express as plt


DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True, parents=True)

RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(exist_ok=True)

PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)


In [ ]:
spreadsheet_path_list = list(RAW_DIR.rglob('*.txt'))
print(len(spreadsheet_path_list), 'record files found')

# sp_path = sorted(spreadsheet_path_list, key=lambda x: str(x))[10]
# print(f'Processing {sp_path}')
# sp_filename = sp_path.name


In [ ]:

def read_quartus_signaltap_into_df(filepath: Path) -> pd.DataFrame:
    rec_text_raw = filepath.read_text()
    rec_col_names = [
        raw_line.split('|')[-1]
        for raw_line in rec_text_raw[:rec_text_raw.find('Data Table:')].splitlines()
        if re.match(r'^\d', raw_line)
    ]
    rec_raw_text = re.sub(r' +', ',', '\n'.join([raw_line.strip() for raw_line in rec_text_raw.split('sample\n')[1].splitlines()]))
    # print(rec_col_names)
    # print(rec_raw_text[:1000])

    rec_raw_pd_data = f'idx,{",".join(rec_col_names)}\n' + rec_raw_text
    raw_rec_pd = pd.read_csv(StringIO(rec_raw_pd_data), skiprows=0, header=0)
    rec_pd = raw_rec_pd.drop(columns=['idx'])
    return rec_pd


In [ ]:
rec_df = read_quartus_signaltap_into_df(spreadsheet_path_list[11])
rec_df


In [ ]:

# Convert dataframe into VCD file

from vcd import VCDWriter


def convert_df_to_vcd(record: pd.DataFrame, target_file: Path):
    with target_file.open('w') as fl:
        with VCDWriter(fl, timescale="1 ns", date="today") as writer:
            
            signals = {}
            for col in record.columns:
                signals[col] = writer.register_var(
                    scope="top", 
                    name=col, 
                    var_type="wire", 
                    size=1
                )

            last_states = {col: None for col in record.columns}
            
            for timestamp, row in record.iterrows():
                for col in record.columns:
                    current_val = int(row[col])
                    
                    if current_val != last_states[col]:
                        writer.change(signals[col], timestamp, current_val)
                        last_states[col] = current_val


convert_df_to_vcd(rec_df, (RAW_DIR / 'result').with_suffix('.vcd'))


In [ ]:
# record = list(map(lambda x: twos_complement(x[:-1], 16), (rec_pd[ 2**13 : ].to_list())))
cur_record = rec_df.iloc[ 2**14 : , :].iloc[:4000, 2]

print(cur_record)
plt.line(cur_record)


In [ ]:
def shrink_channel_with_major_vote():
    pass


In [ ]:
chan_list = rec_df['spi_mosi'].to_list()
